# Импорты и конфигурация

In [1]:
import os
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import os, cv2, glob, random, torch, numpy as np, matplotlib.pyplot as plt, csv
import seaborn as sns
from PIL import Image, ImageDraw
# import mediapipe as mp
from facenet_pytorch import MTCNN
from torchvision import transforms
from tqdm import tqdm
from random import sample
from collections import defaultdict
from sklearn.preprocessing import label_binarize
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, matthews_corrcoef, roc_curve, auc,
    average_precision_score, precision_recall_curve
)
import timm

In [2]:
REAL_DATASET = "/home/vladix35/Diploma/Datasets/FaceForensics++_C23/original"
FAKE_DATASET = "/home/vladix35/Diploma/Datasets/FaceForensics++_C23/Deepfakes"
MAIN_OUTPUT_FOLDER = "/home/vladix35/Diploma/output"

MODEL = "Xception"

FRAMES_PER_VIDEO = 30
BATCH_SIZE = 8
NUM_EPOCHS = 5
LEARNING_RATE = 1e-4

TRAIN_RATIO = 0.75
VAL_RATIO = 0.125
TEST_RATIO = 0.125

RANDOM_SEED = 42

# Загрузка данных:

In [3]:
_device = torch.device("cuda")
_mtcnn = MTCNN(keep_all=False, device=_device)

In [4]:
def _detect_face_bbox(frame: np.ndarray):
    """(x1, y1, x2, y2) or None."""
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # for det in _mp_detectors:
    #     res = det.process(rgb)
    #     if res.detections:
    #         bb = res.detections[0].location_data.relative_bounding_box
    #         h, w, _ = frame.shape
    #         return bb.xmin * w, bb.ymin * h, (bb.xmin + bb.width) * w, (bb.ymin + bb.height) * h

    boxes, _ = _mtcnn.detect(rgb)
    if boxes is not None:
        return tuple(boxes[0])
    return None

In [5]:
def crop_face_from_frame(frame: np.ndarray, fallback_size=(224, 224), margin=0.2):
    """Return PIL.Image of the face crop or None."""
    bbox = _detect_face_bbox(frame)
    if bbox is None:
        return None
    x1, y1, x2, y2 = bbox
    h, w, _ = frame.shape
    dw, dh = (x2 - x1) * margin, (y2 - y1) * margin
    x1, y1 = int(max(x1 - dw, 0)),            int(max(y1 - dh, 0))
    x2, y2 = int(min(x2 + dw, w)),            int(min(y2 + dh, h))

    face = frame[y1:y2, x1:x2]
    if face.size == 0:
        return None
    return Image.fromarray(cv2.resize(face, fallback_size))

In [6]:
def sample_frames_fixed_interval(video_path: str, frames_per_video: int = FRAMES_PER_VIDEO):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    if total <= 0:
        return []
    step = max(total // frames_per_video, 1)
    return [i * step for i in range(frames_per_video) if i * step < total]

In [7]:
class DeepFakeFrameDataset(Dataset):
    """
    Returns (tensor_image, label, video_path).
    If `face_crop=True` tries to crop a face; otherwise uses the whole frame.
    """

    def __init__(self, frame_info_list, transform=None, face_crop=True):
        self.frame_info_list = frame_info_list
        self.transform = transform
        self.face_crop = face_crop

    def __len__(self):
        return len(self.frame_info_list)

    def __getitem__(self, idx):
        video_path, frame_num, label = self.frame_info_list[idx]
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        frame_pil, attempt, ok = None, 0, False
        while attempt < 5 and frame_pil is None:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
            ok, frame = cap.read()
            if not ok:
                break

            if self.face_crop:
                frame_pil = crop_face_from_frame(frame)
            else:
                frame_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

            if frame_pil is None:
                frame_num = random.randint(0, max(total_frames - 1, 0))
                attempt += 1
        cap.release()

        if frame_pil is None and ok:
            frame_pil = Image.fromarray(cv2.resize(frame, (224, 224)))

        if self.transform:
            frame_pil = self.transform(frame_pil)
        return frame_pil, torch.tensor(label, dtype=torch.long), video_path

In [8]:
def _gather_frames(video_list, require_face):
    info = []
    for vid_path, label in tqdm(video_list, desc="Processing videos", unit="video"):
        for idx in sample_frames_fixed_interval(vid_path):
            if not require_face:
                info.append((vid_path, idx, label))
                continue
            cap = cv2.VideoCapture(vid_path)
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ok, frame = cap.read()
            cap.release()
            if ok and crop_face_from_frame(frame) is not None:
                info.append((vid_path, idx, label))
    return info

In [9]:
_aug_resize_256 = transforms.Resize((256, 256))
_to_tensor_norm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

FACE_TRAIN_TRANSFORM = transforms.Compose([
    _aug_resize_256,
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    _to_tensor_norm,
])

FRAME_TRAIN_TRANSFORM = transforms.Compose([
    _aug_resize_256,
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    _to_tensor_norm,
])

VAL_TEST_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    _to_tensor_norm,
])

In [10]:
def prepare_data_faces(real_dataset_path: str, fake_dataset_path: str):
    """
    Face-cropped frames pipeline (exactly your original behaviour).
    Returns: train_ds, val_ds, test_ds
    """
    random.seed(RANDOM_SEED)
    real_videos = sorted(glob.glob(os.path.join(real_dataset_path, "*.mp4")))
    fake_videos = sorted(glob.glob(os.path.join(fake_dataset_path, "*.mp4")))
    
    vids = [(v, 0) for v in real_videos] + [(v, 1) for v in fake_videos]
    random.shuffle(vids)

    n_total = len(vids)
    n_train, n_val = int(TRAIN_RATIO * n_total), int(VAL_RATIO * n_total)
    train_v, val_v, test_v = vids[:n_train], vids[n_train:n_train+n_val], vids[n_train+n_val:]
    tqdm.write(f"[faces] split: {len(train_v)} train / {len(val_v)} val / {len(test_v)} test")

    train_info = _gather_frames(train_v, require_face=True)
    val_info   = _gather_frames(val_v,   require_face=True)
    test_info  = _gather_frames(test_v,  require_face=True)
    tqdm.write(f"[faces] frames: train={len(train_info)} val={len(val_info)} test={len(test_info)}")

    train_ds = DeepFakeFrameDataset(train_info, FACE_TRAIN_TRANSFORM, face_crop=True)
    val_ds   = DeepFakeFrameDataset(val_info,   VAL_TEST_TRANSFORM,  face_crop=True)
    test_ds  = DeepFakeFrameDataset(test_info,  VAL_TEST_TRANSFORM,  face_crop=True)
    return train_ds, val_ds, test_ds

In [11]:
def prepare_data_full_frame(real_dataset_path: str, fake_dataset_path: str):
    """
    Full-frame pipeline (no face detection, faster).
    Returns: train_ds, val_ds, test_ds
    """
    random.seed(RANDOM_SEED)
    real_videos = sorted(glob.glob(os.path.join(real_dataset_path, "*.mp4")))
    fake_videos = sorted(glob.glob(os.path.join(fake_dataset_path, "*.mp4")))

    vids = [(v, 0) for v in real_videos] + [(v, 1) for v in fake_videos]
    random.shuffle(vids)

    n_total = len(vids)
    n_train, n_val = int(TRAIN_RATIO * n_total), int(VAL_RATIO * n_total)
    train_v, val_v, test_v = vids[:n_train], vids[n_train:n_train+n_val], vids[n_train+n_val:]
    tqdm.write(f"[frames] split: {len(train_v)} train / {len(val_v)} val / {len(test_v)} test")

    train_info = _gather_frames(train_v, require_face=False)
    val_info   = _gather_frames(val_v,   require_face=False)
    test_info  = _gather_frames(test_v,  require_face=False)
    tqdm.write(f"[frames] frames: train={len(train_info)} val={len(val_info)} test={len(test_info)}")

    train_ds = DeepFakeFrameDataset(train_info, FRAME_TRAIN_TRANSFORM, face_crop=False)
    val_ds   = DeepFakeFrameDataset(val_info,   VAL_TEST_TRANSFORM,    face_crop=False)
    test_ds  = DeepFakeFrameDataset(test_info,  VAL_TEST_TRANSFORM,    face_crop=False)
    return train_ds, val_ds, test_ds

In [12]:
device = torch.device("cuda")
tqdm.write(f"Device: {device}")

Device: cuda


# Подготовка нашей модели для обучения на основе Xception:

In [13]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y

In [14]:
def validate_video_level(model, val_loader, device):
    """Video‐level validation with multiple aggregation methods"""
    model.eval()
    video_dict = defaultdict(lambda: {"labels": [], "probs": [], "preds": []})
    criterion = nn.CrossEntropyLoss()
    running_loss = 0.0
    total = 0

    with torch.no_grad():
        for images, labels, video_paths in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            total += images.size(0)

            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)
            for i, vp in enumerate(video_paths):
                video_dict[vp]["labels"].append(labels[i].item())
                video_dict[vp]["probs"].append(probs[i].cpu().numpy())
                video_dict[vp]["preds"].append(preds[i].item())

    val_loss = running_loss / total if total else 0.0

    methods = {
        'mean_prob':         lambda x: np.mean([p[1] for p in x]),
        'max_prob':          lambda x: np.max([p[1] for p in x]),
        'min_prob':          lambda x: np.min([p[1] for p in x]),
        'median_prob':       lambda x: np.median([p[1] for p in x]),
        'trimmed_mean_prob': lambda x: np.mean(sorted([p[1] for p in x])[1:-1]) if len(x)>2 else np.mean([p[1] for p in x]),
        'vote_percentage':   lambda x: np.mean(x)
    }

    metrics_all = {}
    best_f1 = -1.0
    best_acc = 0.0
    best_method = None

    for name, agg in methods.items():
        y_true, y_pred, scores = [], [], []
        for info in video_dict.values():
            if not info["labels"]:
                continue
            true = info["labels"][0]
            score = agg(info["preds"] if name=='vote_percentage' else info["probs"])
            pred = 1 if score >= 0.5 else 0
            y_true.append(true)
            y_pred.append(pred)
            scores.append(score)

        if not y_true:
            continue

        acc  = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec  = recall_score(y_true, y_pred, zero_division=0)
        f1   = f1_score(y_true, y_pred, zero_division=0)
        mcc  = matthews_corrcoef(y_true, y_pred)
        y_bin = np.array(y_true)
        fpr, tpr, _ = roc_curve(y_bin, scores)
        eer     = fpr[np.nanargmin(np.abs((1-tpr)-fpr))]
        roc_auc = auc(fpr, tpr)
        ap      = average_precision_score(y_bin, scores)

        metrics_all[name] = {
            'accuracy': acc, 'precision': prec,
            'recall': rec, 'f1': f1,
            'mcc': mcc, 'auc': roc_auc,
            'ap': ap, 'eer': eer
        }

        if f1 > best_f1:
            best_f1 = f1
            best_acc = acc
            best_method = name

    return val_loss, best_acc, best_f1, best_method, metrics_all

In [15]:
def train_model(model, train_loader, val_loader, device, output_folder):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LEARNING_RATE, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.1, patience=2
    )
    scaler = GradScaler()
    early_stop = 5
    no_improve = 0
    best_val_loss = float('inf')
    model_path = os.path.join(output_folder, "Xception.pth")

    if os.path.exists(model_path):
        tqdm.write("Xception already trained; loading.")
        model.load_state_dict(torch.load(model_path))
        metrics_file = os.path.join(output_folder, "Xception_validation_metrics.txt")
        best_method = "mean_prob"  # По умолчанию
        if os.path.exists(metrics_file):
            with open(metrics_file, "r") as f:
                for line in f:
                    if line.startswith("Best:"):
                        best_method = line.split(":")[1].strip()
                        break
        
        return model, best_method

    train_losses, train_accs = [], []
    val_losses, val_accs, val_f1s = [], [], []

    for epoch in range(NUM_EPOCHS):
        # --- train ---
        model.train()
        running, corr, tot = 0.0, 0, 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", leave=False)
        for imgs, lbls, _ in pbar:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            with autocast():
                outs = model(imgs)
                loss = criterion(outs, lbls)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running += loss.item() * imgs.size(0)
            preds = outs.argmax(dim=1)
            corr += (preds==lbls).sum().item()
            tot  += lbls.size(0)
            pbar.set_postfix(loss=f"{running/tot:.4f}", acc=f"{corr/tot:.4f}")

        train_losses.append(running/tot)
        train_accs.append(corr/tot)

        # --- validate ---
        val_loss, val_acc, val_f1, best_method, metrics_all = validate_video_level(
            model, val_loader, device
        )
        scheduler.step(val_loss)
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        val_f1s.append(val_f1)

        tqdm.write(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
                     f"Train loss {train_losses[-1]:.4f}, acc {train_accs[-1]:.4f} | "
                     f"Val loss {val_loss:.4f}, acc {val_acc:.4f}, f1 {val_f1:.4f} ({best_method})")

        out_txt = os.path.join(output_folder, "Xception_validation_metrics.txt")
        with open(out_txt, "a") as f:
            f.write(f"\nEpoch {epoch+1}\n")
            for m, met in metrics_all.items():
                f.write(f"{m} -> Acc: {met['accuracy']:.4f}, "
                        f"Prec: {met['precision']:.4f}, Rec: {met['recall']:.4f}, "
                        f"F1: {met['f1']:.4f}, MCC: {met['mcc']:.4f}, "
                        f"AUC: {met['auc']:.4f}, AP: {met['ap']:.4f}, EER: {met['eer']:.4f}\n")
            f.write(f"Best: {best_method}\n")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_method_overall = best_method
            no_improve = 0
            torch.save(model.state_dict(), model_path)
            tqdm.write(f"Saved new best model (loss={val_loss:.4f})")
        else:
            no_improve += 1
            if no_improve >= early_stop:
                tqdm.write("Early stopping triggered.")
                break

    # ---- PLOT TRAINING CURVES ----
    plt.figure()
    plt.plot(range(1, len(train_losses)+1), train_losses, label='Train Loss')
    plt.plot(range(1, len(val_losses)+1),   val_losses,   label='Val Loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.title('Loss Curves'); plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, 'training_loss.png'))
    plt.close()

    plt.figure()
    plt.plot(range(1, len(train_accs)+1), train_accs, label='Train Acc')
    plt.plot(range(1, len(val_accs)+1),   val_accs,   label='Val Acc')
    plt.xlabel('Epoch'); plt.ylabel('Accuracy')
    plt.title('Accuracy Curves'); plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, 'training_accuracy.png'))
    plt.close()

    plt.figure()
    plt.plot(range(1, len(val_f1s)+1), val_f1s, label='Val F1')
    plt.xlabel('Epoch'); plt.ylabel('F1 Score')
    plt.title('F1 Score Curve'); plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, 'validation_f1.png'))
    plt.close()

    # ---- PLOT F1 vs LOSS ----
    plt.figure()
    plt.plot(val_losses, val_f1s, marker='o')
    plt.xlabel('Validation Loss')
    plt.ylabel('F1 Score')
    plt.title('F1 vs Loss')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, 'f1_vs_loss.png'))
    plt.close()

    return model, best_method_overall

In [16]:
def Xception_main(train_ds, val_ds, test_ds, device, output_folder):
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    backbone = timm.create_model('xception', pretrained=True)

    for p in backbone.parameters():
        p.requires_grad = False

    unfreeze_modules = ['block6','block7','block8','conv4','bn4']
    for name, module in backbone.named_children():
        if name in unfreeze_modules:
            for p in module.parameters():
                p.requires_grad = True

    class XceptionSE(nn.Module):
        def __init__(self, base):
            super().__init__()
            self.base = base
            self.se   = SEBlock(channels=2048, reduction=16)
            self.fc   = nn.Linear(2048, 2)
        def forward(self, x):
            f = self.base.forward_features(x)
            f = self.se(f)                     
            p = self.base.global_pool(f)      
            p = torch.flatten(p, 1)           
            return self.fc(p)                

    model = XceptionSE(backbone).to(device)

    model, best_method = train_model(model, train_loader, val_loader, device, output_folder)
    return test_loader, model, best_method

In [17]:
def _collect_frame_predictions(model, test_loader, device):
    model.eval()
    video_dict = defaultdict(lambda: {"labels": [], "probs": [], "preds": []})

    with torch.no_grad():
        for imgs, labels, video_paths in tqdm(test_loader, desc="Testing"):
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)
            for i, vpath in enumerate(video_paths):
                video_dict[vpath]["labels"].append(labels[i].item())
                video_dict[vpath]["probs"].append(probs[i].cpu().numpy())
                video_dict[vpath]["preds"].append(preds[i].item())

    return video_dict


In [18]:
def _aggregate_video_predictions(video_dict, method_name):
    """Return metrics dict."""
    agg_fn = {
        "mean_prob":         lambda x: np.mean([p[1] for p in x]),
        "max_prob":          lambda x: np.max([p[1] for p in x]),
        "min_prob":          lambda x: np.min([p[1] for p in x]),
        "median_prob":       lambda x: np.median([p[1] for p in x]),
        "trimmed_mean_prob": lambda x: np.mean(sorted([p[1] for p in x])[1:-1])
                                    if len(x) > 2 else np.mean([p[1] for p in x]),
        "vote_percentage":   lambda x: np.mean(x),
    }[method_name]

    v_labels, v_preds, v_scores = [], [], []
    error_records, correct_records = [], []

    for vpath, info in video_dict.items():
        true = info["labels"][0]
        score = agg_fn(info["preds"] if method_name == "vote_percentage"
                       else info["probs"])
        pred = 1 if score >= .5 else 0

        v_labels.append(true);  v_preds.append(pred);  v_scores.append(score)

        cap = cv2.VideoCapture(vpath); total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
        row = {
            "video_name": os.path.basename(vpath),
            "total_frames": total,
            "true_label": true,
            "predicted_label": pred,
            "method": method_name,
            "confidence_score": round(score, 4),
            "result_type": ("TP" if pred == 1 else "TN") if pred == true
                           else ("FP" if pred == 1 else "FN"),
        }
        (correct_records if pred == true else error_records).append(row)

    a = accuracy_score(v_labels, v_preds)
    p = precision_score(v_labels, v_preds, zero_division=0)
    r = recall_score(v_labels, v_preds, zero_division=0)
    f1 = f1_score(v_labels, v_preds, zero_division=0)
    mcc = matthews_corrcoef(v_labels, v_preds)
    y_bin = label_binarize(v_labels, classes=[0, 1]).ravel()
    fpr, tpr, _ = roc_curve(y_bin, v_scores)
    eer = fpr[np.nanargmin(np.abs((1 - tpr) - fpr))]
    ap = average_precision_score(y_bin, v_scores)

    return {
        "accuracy": a, "precision": p, "recall": r, "f1": f1,
        "mcc": mcc, "ap": ap, "eer": eer,
        "labels": v_labels, "preds": v_preds, "scores": v_scores,
        "errors": error_records, "corrects": correct_records,
    }

In [19]:
def _log_metrics(metrics_all, best_method, MODEL, out_dir):
    with open(os.path.join(out_dir, f"{MODEL}_metrics.txt"), "w") as f:
        f.write("Video-level metrics per aggregation method:\n\n")
        for name, met in metrics_all.items():
            f.write(f"Method: {name}\n")
            f.write(f"  Accuracy : {met['accuracy']:.4f}\n")
            f.write(f"  Precision: {met['precision']:.4f}\n")
            f.write(f"  Recall   : {met['recall']:.4f}\n")
            f.write(f"  F1       : {met['f1']:.4f}\n")
            f.write(f"  MCC      : {met['mcc']:.4f}\n")
            f.write(f"  AP       : {met['ap']:.4f}\n")
            f.write(f"  EER      : {met['eer']:.4f}\n\n")
        f.write(f"Best aggregation (by F1): {best_method}\n")

In [20]:
def _write_csv(rows, name, MODEL, out_dir):
    path = os.path.join(out_dir, f"{MODEL}_{name}.csv")
    if not rows:
        tqdm.write(f"No records for {name}.")
        return
    with open(path, "w", newline="") as f:
        hdr = ["video_name","total_frames","true_label","predicted_label",
               "method","confidence_score","result_type"]
        csv.DictWriter(f, hdr).writeheader();  csv.DictWriter(f, hdr).writerows(rows)
    tqdm.write(f"Saved {name} to {path}")

In [21]:
def get_target_layer(model, MODEL):
    """Get target layer for Grad-CAM based on model architecture"""
    if MODEL == "ResNet50":
        return [model.layer4[-1]]
    elif MODEL == "Xception":
        return [model.base.conv4]
    elif MODEL == "SwinTransformer":
        return [model.backbone.layers[-1].blocks[-1].norm1]
    else:
        return [list(model.children())[-2]] 

In [22]:
def generate_heatmap_grid(model, MODEL, device, video_path, transform, face_crop):
    """
    Generate a grid of heatmaps for a video (5x6 grid of frames)
    Returns: PIL Image of the grid
    """
    frame_indices = sample_frames_fixed_interval(video_path)
    cap = cv2.VideoCapture(video_path)
    frames = []
    
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            continue
            
        if face_crop:
            frame_pil = crop_face_from_frame(frame)
            if frame_pil is None:
                continue
            orig_frame = np.array(frame_pil)
        else:
            orig_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            orig_frame = cv2.resize(orig_frame, (224, 224))
            frame_pil = Image.fromarray(orig_frame)
        
        input_tensor = transform(frame_pil).unsqueeze(0).to(device)
        frames.append((orig_frame, input_tensor, idx))
    
    cap.release()
    
    if not frames:
        return None
    
    target_layer = get_target_layer(model, MODEL)
    cam = GradCAM(model=model, target_layers=target_layer)
    
    heatmap_images = []
    percentages = []
    
    for orig_frame, input_tensor, idx in frames:
        with torch.no_grad():
            output = model(input_tensor)
            prob = torch.softmax(output, dim=1)[0]
            pred_class = torch.argmax(output).item()
            pred_percent = prob[pred_class].item() * 100
        
        grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0, :]
        
        frame_norm = orig_frame.astype(np.float32) / 255
        
        visualization = show_cam_on_image(
            frame_norm, 
            grayscale_cam,
            use_rgb=True
        )
        heatmap_images.append(visualization)
        percentages.append(f"{pred_percent:.1f}%")
    
    grid_img = Image.new('RGB', (6 * 224, 5 * (224 + 30)), color='white')
    draw = ImageDraw.Draw(grid_img)
    
    for i, (img, percent) in enumerate(zip(heatmap_images, percentages)):
        if i >= 30: 
            break
            
        row = i // 6
        col = i % 6
        
        img_pil = Image.fromarray(img)
        
        x = col * 224
        y = row * (224 + 30)
        
        grid_img.paste(img_pil, (x, y))
        
        bbox = draw.textbbox((0, 0), percent)
        text_width = bbox[2] - bbox[0]
        draw.text(
            (x + (224 - text_width) // 2, y + 224 + 5),
            percent,
            fill='black'
        )
    
    if hasattr(cam, 'activations_and_grads'):
        cam.activations_and_grads.release()
    if hasattr(cam, 'model'):
        cam.model.zero_grad()
    
    return grid_img

In [23]:
def _test_model(model, test_loader, device, MODEL, out_dir, face_crop, agg_method=None):
    os.makedirs(out_dir, exist_ok=True)
    video_dict = _collect_frame_predictions(model, test_loader, device)

    valid_methods = [
        "mean_prob", "max_prob", "min_prob",
        "median_prob", "trimmed_mean_prob", "vote_percentage"
    ]

    if agg_method not in valid_methods:
        raise ValueError(f"Invalid aggregation method '{agg_method}'. Must be one of: {valid_methods}")

    metrics = _aggregate_video_predictions(video_dict, agg_method)
    metrics_all = {agg_method: metrics}

    _log_metrics(metrics_all, agg_method, MODEL, out_dir)
    _write_csv(metrics["errors"],  "errors",  MODEL, out_dir)
    _write_csv(metrics["corrects"], "correct", MODEL, out_dir)

    labels_bin = label_binarize(metrics["labels"], classes=[0, 1]).ravel()
    cm = confusion_matrix(metrics["labels"], metrics["preds"])
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Real", "Fake"], yticklabels=["Real", "Fake"])
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.title(f"Confusion Matrix ({agg_method})")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{MODEL}_confusion_matrix.png"))
    plt.close()

    fpr, tpr, _ = roc_curve(labels_bin, metrics["scores"])
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC = {auc(fpr, tpr):.2f}")
    plt.plot([0, 1], [0, 1], "k--")
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title("ROC Curve"); plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{MODEL}_roc_curve.png"))
    plt.close()

    prec, rec, _ = precision_recall_curve(labels_bin, metrics["scores"])
    plt.figure(figsize=(6, 5))
    plt.plot(rec, prec, label=f"AP = {metrics['ap']:.2f}")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title("Precision-Recall Curve"); plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{MODEL}_pr_curve.png"))
    plt.close()

    cam_dir = os.path.join(out_dir, 'cams')
    os.makedirs(cam_dir, exist_ok=True)
    categories = ['FN', 'FP', 'TP', 'TN']
    for cat in categories:
        os.makedirs(os.path.join(cam_dir, cat), exist_ok=True)

    selected_videos = {cat: [] for cat in categories}
    for record in metrics["errors"] + metrics["corrects"]:
        cat = record['result_type']
        if len(selected_videos[cat]) < 3:
            selected_videos[cat].append(record)

    for cat, records in selected_videos.items():
        for record in records:
            video_path = os.path.join(
                REAL_DATASET if record['true_label'] == 0 else FAKE_DATASET,
                record['video_name']
            )
            if not os.path.exists(video_path):
                tqdm.write(f"Video not found: {video_path}")
                continue

            grid = generate_heatmap_grid(
                model=model,
                MODEL=MODEL,
                device=device,
                video_path=video_path,
                transform=VAL_TEST_TRANSFORM,
                face_crop=face_crop
            )

            if grid:
                output_path = os.path.join(cam_dir, cat, f"{record['video_name']}.png")
                grid.save(output_path)
                tqdm.write(f"Saved heatmap grid for {record['video_name']} to {output_path}")

In [24]:
def test_model_faces(model, test_loader, device, MODEL, output_folder, agg_method=None):
    """Evaluation routine for face‐cropped data."""
    _test_model(model, test_loader, device, MODEL,
                output_folder, face_crop=True, agg_method=agg_method)

def test_model_full_frame(model, test_loader, device, MODEL, output_folder, agg_method=None):
    """Evaluation routine for full‐frame data."""
    _test_model(model, test_loader, device, MODEL,
                output_folder, face_crop=False, agg_method=agg_method)
    
def run_one_test(model, test_loader, test_fn,
                 device, MODEL, test_out, agg_method=None):
    os.makedirs(test_out, exist_ok=True)
    test_fn(model, test_loader, device, MODEL, test_out, agg_method=agg_method)

In [25]:
# ---------------------------
# PHASE 1: PREPARE “faces” ONCE
# ---------------------------
tqdm.write("===== PHASE 1: PREPARING ‘faces’ DATA … =====")
train_ds_faces, val_ds_faces, test_ds_faces = prepare_data_faces(
    REAL_DATASET, FAKE_DATASET
)
tqdm.write("Preparing full-frame test data...")
_, _, test_ds_frames_for_phase1 = prepare_data_full_frame(
    REAL_DATASET, FAKE_DATASET
)

===== PHASE 1: PREPARING ‘faces’ DATA … =====
[faces] split: 1500 train / 250 val / 250 test


Processing videos: 100%|██████████| 250/250 [17:05<00:00,  4.10s/video]


[faces] frames: train=44985 val=7499 test=7500
Preparing full-frame test data...
[frames] split: 1500 train / 250 val / 250 test


Processing videos: 100%|██████████| 250/250 [00:01<00:00, 181.73video/s]

[frames] frames: train=45000 val=7500 test=7500


In [26]:
# ---------------------------
# PHASE 1: PREPARE “faces” ONCE
# ---------------------------
tqdm.write(f"========== PHASE 1 (FACES) → MODEL: {MODEL} ==========")

training_out = os.path.join(MAIN_OUTPUT_FOLDER, MODEL, "faces", "training")
os.makedirs(training_out, exist_ok=True)

test_loader_faces, model, best_method_faces = Xception_main(
    train_ds_faces,
    val_ds_faces,
    test_ds_faces,
    device,
    training_out
)

combo_name = "faces_on_faces"
tqdm.write(f"    └─▶ TESTING {combo_name} …")
test_out_dir = os.path.join(MAIN_OUTPUT_FOLDER, MODEL, combo_name, "testing")
run_one_test(model, test_loader_faces, test_model_faces, device, MODEL, test_out_dir, agg_method=best_method_faces)

combo_name = "faces_on_frames"
tqdm.write(f"    └─▶ TESTING {combo_name} …")
frames_test_loader = DataLoader(
    test_ds_frames_for_phase1,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)
test_out_dir = os.path.join(MAIN_OUTPUT_FOLDER, MODEL, combo_name, "testing")
run_one_test(model, frames_test_loader, test_model_full_frame,
                device, MODEL, test_out_dir, agg_method=best_method_faces)

/home/vladix35/anaconda3/envs/myenv/lib/python3.10/site-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


========== PHASE 1 (FACES) → MODEL: Xception ==========
Xception already trained; loading.
    └─▶ TESTING faces_on_faces …


Testing: 100%|██████████| 938/938 [18:11<00:00,  1.16s/it]


Saved errors to /home/vladix35/Diploma/output/Xception/faces_on_faces/testing/Xception_errors.csv
Saved correct to /home/vladix35/Diploma/output/Xception/faces_on_faces/testing/Xception_correct.csv
Saved heatmap grid for 568_628.mp4 to /home/vladix35/Diploma/output/Xception/faces_on_faces/testing/cams/FN/568_628.mp4.png
Saved heatmap grid for 025_067.mp4 to /home/vladix35/Diploma/output/Xception/faces_on_faces/testing/cams/FN/025_067.mp4.png
Saved heatmap grid for 880_135.mp4 to /home/vladix35/Diploma/output/Xception/faces_on_faces/testing/cams/FN/880_135.mp4.png
Saved heatmap grid for 090_086.mp4 to /home/vladix35/Diploma/output/Xception/faces_on_faces/testing/cams/TP/090_086.mp4.png
Saved heatmap grid for 557_560.mp4 to /home/vladix35/Diploma/output/Xception/faces_on_faces/testing/cams/TP/557_560.mp4.png
Saved heatmap grid for 671_677.mp4 to /home/vladix35/Diploma/output/Xception/faces_on_faces/testing/cams/TP/671_677.mp4.png
Saved heatmap grid for 141.mp4 to /home/vladix35/Diploma/o

Testing: 100%|██████████| 938/938 [11:59<00:00,  1.30it/s]


Saved errors to /home/vladix35/Diploma/output/Xception/faces_on_frames/testing/Xception_errors.csv
Saved correct to /home/vladix35/Diploma/output/Xception/faces_on_frames/testing/Xception_correct.csv
Saved heatmap grid for 090_086.mp4 to /home/vladix35/Diploma/output/Xception/faces_on_frames/testing/cams/FN/090_086.mp4.png
Saved heatmap grid for 671_677.mp4 to /home/vladix35/Diploma/output/Xception/faces_on_frames/testing/cams/FN/671_677.mp4.png
Saved heatmap grid for 932_384.mp4 to /home/vladix35/Diploma/output/Xception/faces_on_frames/testing/cams/FN/932_384.mp4.png
Saved heatmap grid for 995.mp4 to /home/vladix35/Diploma/output/Xception/faces_on_frames/testing/cams/FP/995.mp4.png
Saved heatmap grid for 490.mp4 to /home/vladix35/Diploma/output/Xception/faces_on_frames/testing/cams/FP/490.mp4.png
Saved heatmap grid for 039.mp4 to /home/vladix35/Diploma/output/Xception/faces_on_frames/testing/cams/FP/039.mp4.png
Saved heatmap grid for 568_628.mp4 to /home/vladix35/Diploma/output/Xcepti

In [26]:
# ----------------------------
# PHASE 2: PREPARE “frames” ONCE
# ----------------------------
tqdm.write("===== PHASE 2: PREPARING ‘frames’ DATA … =====")
train_ds_frames, val_ds_frames, test_ds_frames = prepare_data_full_frame(
    REAL_DATASET, FAKE_DATASET
)

faces_test_loader_for_phase2 = DataLoader(
    test_ds_faces,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

tqdm.write(f"========== PHASE 2 (FRAMES) → MODEL: {MODEL} ==========")

training_out = os.path.join(MAIN_OUTPUT_FOLDER, MODEL, "frames", "training")
os.makedirs(training_out, exist_ok=True)

test_loader_frames, model, best_method_frames = Xception_main(
    train_ds_frames,
    val_ds_frames,
    test_ds_frames,
    device,
    training_out
)

combo_name = "frames_on_faces"
tqdm.write(f"    └─▶ TESTING {combo_name} …")
test_out_dir = os.path.join(MAIN_OUTPUT_FOLDER, MODEL, combo_name, "testing")
run_one_test(model, faces_test_loader_for_phase2, test_model_faces,
             device, MODEL, test_out_dir, agg_method=best_method_frames)

combo_name = "frames_on_frames"
tqdm.write(f"    └─▶ TESTING {combo_name} …")
test_out_dir = os.path.join(MAIN_OUTPUT_FOLDER, MODEL, combo_name, "testing")
run_one_test(model, test_loader_frames, test_model_full_frame,
             device, MODEL, test_out_dir, agg_method=best_method_frames)

===== PHASE 2: PREPARING ‘frames’ DATA … =====
[frames] split: 1500 train / 250 val / 250 test


Processing videos: 100%|██████████| 250/250 [00:01<00:00, 189.10video/s]
/home/vladix35/anaconda3/envs/myenv/lib/python3.10/site-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


[frames] frames: train=45000 val=7500 test=7500
========== PHASE 2 (FRAMES) → MODEL: Xception ==========
Xception already trained; loading.
    └─▶ TESTING frames_on_faces …


Testing: 100%|██████████| 938/938 [17:36<00:00,  1.13s/it]


Saved errors to /home/vladix35/Diploma/output/Xception/frames_on_faces/testing/Xception_errors.csv
Saved correct to /home/vladix35/Diploma/output/Xception/frames_on_faces/testing/Xception_correct.csv
Saved heatmap grid for 141.mp4 to /home/vladix35/Diploma/output/Xception/frames_on_faces/testing/cams/FP/141.mp4.png
Saved heatmap grid for 995.mp4 to /home/vladix35/Diploma/output/Xception/frames_on_faces/testing/cams/FP/995.mp4.png
Saved heatmap grid for 175.mp4 to /home/vladix35/Diploma/output/Xception/frames_on_faces/testing/cams/FP/175.mp4.png
Saved heatmap grid for 568_628.mp4 to /home/vladix35/Diploma/output/Xception/frames_on_faces/testing/cams/TP/568_628.mp4.png
Saved heatmap grid for 090_086.mp4 to /home/vladix35/Diploma/output/Xception/frames_on_faces/testing/cams/TP/090_086.mp4.png
Saved heatmap grid for 557_560.mp4 to /home/vladix35/Diploma/output/Xception/frames_on_faces/testing/cams/TP/557_560.mp4.png
Saved heatmap grid for 039.mp4 to /home/vladix35/Diploma/output/Xception/f

Testing: 100%|██████████| 938/938 [11:03<00:00,  1.41it/s]


Saved errors to /home/vladix35/Diploma/output/Xception/frames_on_frames/testing/Xception_errors.csv
Saved correct to /home/vladix35/Diploma/output/Xception/frames_on_frames/testing/Xception_correct.csv
Saved heatmap grid for 723_704.mp4 to /home/vladix35/Diploma/output/Xception/frames_on_frames/testing/cams/FN/723_704.mp4.png
Saved heatmap grid for 998_561.mp4 to /home/vladix35/Diploma/output/Xception/frames_on_frames/testing/cams/FN/998_561.mp4.png
Saved heatmap grid for 898_922.mp4 to /home/vladix35/Diploma/output/Xception/frames_on_frames/testing/cams/FN/898_922.mp4.png
Saved heatmap grid for 141.mp4 to /home/vladix35/Diploma/output/Xception/frames_on_frames/testing/cams/FP/141.mp4.png
Saved heatmap grid for 995.mp4 to /home/vladix35/Diploma/output/Xception/frames_on_frames/testing/cams/FP/995.mp4.png
Saved heatmap grid for 175.mp4 to /home/vladix35/Diploma/output/Xception/frames_on_frames/testing/cams/FP/175.mp4.png
Saved heatmap grid for 568_628.mp4 to /home/vladix35/Diploma/outpu